In [1]:
# Reproducibility and environment information (run this cell first)
import random
import subprocess
import sys

SEED = 266
random.seed(SEED)

try:
    commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], text=True, stderr=subprocess.DEVNULL
    ).strip()
except (subprocess.CalledProcessError, FileNotFoundError):
    commit = "unavailable (run inside a committed Git checkout)"

print(f"Git commit: {commit}")
print(f"Python: {sys.version.split()[0]}")

try:
    import numpy as np
    np.random.seed(SEED)
except ImportError:
    print("NumPy: not installed")

try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU count: {torch.cuda.device_count()}")
        for index in range(torch.cuda.device_count()):
            print(f"GPU {index}: {torch.cuda.get_device_name(index)}")
    else:
        print("GPU: none detected")
except ImportError:
    print("PyTorch: not installed")
    print("CUDA available: unavailable")
    print("GPU: unavailable")

print(f"SEED={SEED}")

Git commit: 6bdef3cab2196e9e482946eb3274a05ca8e4cd2f
Python: 3.9.6
NumPy: not installed
PyTorch: not installed
CUDA available: unavailable
GPU: unavailable
SEED=266


# Part 1 - Multimodal Retrieval-Augmented Generation

This notebook builds a grounded question-answering system over three read-only economics PDFs.

## Current checkpoint

In this step we only initialize the notebook and complete **Task 1.1.1: PDF Text Extraction**. Chunking, figure extraction, embeddings, retrieval, and answer generation are intentionally left for later steps.

Safety rule: DOC3 Appendix C (page 7) is retained for auditing, but it must be marked `low` trust and must never be used as RAG evidence.

## 1. Locate the Part 1 input files

The data path is resolved at runtime instead of using a personal Google Drive path. The code checks, in order:

1. `DATA266_PART1_DIR` if you set it,
2. the repository's `part1_rag/data` folder and the current working directory,
3. common Colab and Kaggle locations.

This lets the same notebook run after a Git clone in Colab or with competition data attached in Kaggle.

In [2]:
import json
import os
from pathlib import Path

REQUIRED_INPUTS = {
    "kb_manifest.json",
    "Document.pdf",
    "Document2_Supplement_A_Outturns_2011-2013.pdf",
    "Document3_Briefing_B_Labour_Prices_Policy.pdf",
}


def contains_required_inputs(folder):
    folder = Path(folder)
    return folder.is_dir() and all((folder / name).is_file() for name in REQUIRED_INPUTS)


def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    return start


def locate_part1_inputs():
    configured = os.environ.get("DATA266_PART1_DIR")
    repo_root = find_repo_root(Path.cwd())
    candidates = [
        Path(configured).expanduser() if configured else None,
        repo_root / "part1_rag" / "data",
        Path.cwd(),
        Path.cwd() / "Part1",
        Path("/content"),
        Path("/content/Part1"),
        Path("/mnt/data"),
    ]
    for candidate in candidates:
        if candidate is not None and contains_required_inputs(candidate):
            return candidate.resolve()

    # Kaggle datasets are nested under /kaggle/input/<dataset-name>/.
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.is_dir():
        for manifest_path in kaggle_root.rglob("kb_manifest.json"):
            if contains_required_inputs(manifest_path.parent):
                return manifest_path.parent.resolve()

    raise FileNotFoundError(
        "Part 1 files were not found. Attach the Kaggle data or set "
        "DATA266_PART1_DIR to the folder containing kb_manifest.json and the three PDFs."
    )


REPO_ROOT = find_repo_root(Path.cwd())
INPUT_DIR = locate_part1_inputs()
print(f"Repository root: {REPO_ROOT}")
print(f"Part 1 input directory: {INPUT_DIR}")

Repository root: /Users/dingyuyao/Documents/Codex/2026-10-17/referenced-chatgpt-conversation-this-is-an/work/data266-lab2-team41
Part 1 input directory: /private/tmp/data266-part1-inputs-20261020


## 2. Read and validate the knowledge-base manifest

The manifest is the source of truth for each document's ID, expected page count, and trust level. We do not infer those values from filenames. The assertions make the notebook stop early if the wrong files or an altered manifest are attached.

In [3]:
with (INPUT_DIR / "kb_manifest.json").open(encoding="utf-8") as file:
    kb_manifest = json.load(file)

documents = {item["doc_id"]: item for item in kb_manifest["documents"]}
assert set(documents) == {"DOC1", "DOC2", "DOC3"}

doc3_overrides = documents["DOC3"].get("section_trust_overrides", [])
assert any(
    7 in override.get("pages", []) and override.get("trust_level") == "low"
    for override in doc3_overrides
), "DOC3 page 7 must be marked low trust in the manifest."

for doc_id, metadata in documents.items():
    print(
        f"{doc_id}: trust={metadata['trust_level']}, "
        f"expected_pages={metadata['pages']}, file={metadata['file']}"
    )

DOC1: trust=high, expected_pages=37, file=Document.pdf
DOC2: trust=high, expected_pages=7, file=Document2_Supplement_A_Outturns_2011-2013.pdf
DOC3: trust=medium, expected_pages=7, file=Document3_Briefing_B_Labour_Prices_Policy.pdf


## 3. Extract text one page at a time

Each output record keeps the metadata that later stages need: `doc_id`, 1-based `page`, `trust_level`, `evidence_allowed`, and extracted `text`. Keeping page boundaries now makes later citations such as `DOC2:p4` possible.

PyMuPDF is installed only if the runtime does not already provide it.

In [4]:
try:
    import fitz  # PyMuPDF
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "PyMuPDF"])
    import fitz

print(f"PyMuPDF: {fitz.VersionBind}")

PyMuPDF: 1.26.5


In [5]:
def trust_for_page(document_metadata, page_number):
    trust_level = document_metadata["trust_level"]
    for override in document_metadata.get("section_trust_overrides", []):
        if page_number in override.get("pages", []):
            trust_level = override["trust_level"]
    return trust_level


def extract_document_pages(input_dir, document_metadata):
    pdf_path = Path(input_dir) / document_metadata["file"]
    records = []

    with fitz.open(pdf_path) as pdf:
        actual_pages = pdf.page_count
        expected_pages = document_metadata["pages"]
        if actual_pages != expected_pages:
            raise ValueError(
                f"{document_metadata['doc_id']} has {actual_pages} pages; "
                f"manifest expects {expected_pages}."
            )

        for page_index, pdf_page in enumerate(pdf):
            page_number = page_index + 1
            trust_level = trust_for_page(document_metadata, page_number)
            records.append(
                {
                    "doc_id": document_metadata["doc_id"],
                    "page": page_number,
                    "trust_level": trust_level,
                    "evidence_allowed": trust_level != "low",
                    "text": pdf_page.get_text("text", sort=True).strip(),
                }
            )

    return records


## 4. Run extraction and verify the safety invariant

The final checks verify all 51 pages, confirm that no page is empty, and prove that DOC3 page 7 is the only low-trust page. A short preview is printed for inspection; Chunking starts in the next checkpoint, not here.

In [6]:
page_records = []
for doc_id in ("DOC1", "DOC2", "DOC3"):
    records = extract_document_pages(INPUT_DIR, documents[doc_id])
    page_records.extend(records)
    character_count = sum(len(record["text"]) for record in records)
    empty_pages = [record["page"] for record in records if not record["text"]]
    print(
        f"{doc_id}: extracted_pages={len(records)}, "
        f"characters={character_count:,}, empty_pages={empty_pages}"
    )

assert len(page_records) == 51
assert all(record["text"] for record in page_records), "Every PDF page should contain extractable text."

blocked_pages = [
    (record["doc_id"], record["page"], record["trust_level"])
    for record in page_records
    if not record["evidence_allowed"]
]
assert blocked_pages == [("DOC3", 7, "low")]

print(f"Total extracted pages: {len(page_records)}")
print(f"Pages blocked from RAG evidence: {blocked_pages}")

for doc_id in ("DOC1", "DOC2", "DOC3"):
    first_page = next(record for record in page_records if record["doc_id"] == doc_id)
    preview = " ".join(first_page["text"].split())[:180]
    print(f"{doc_id} page 1 preview: {preview}...")

DOC1: extracted_pages=37, characters=176,179, empty_pages=[]
DOC2: extracted_pages=7, characters=10,424, empty_pages=[]
DOC3: extracted_pages=7, characters=8,279, empty_pages=[]
Total extracted pages: 51
Pages blocked from RAG evidence: [('DOC3', 7, 'low')]
DOC1 page 1 preview: 1-1 The Crisis Table 1-1 gives you output growth rates for the world economy, for advanced econ- omies and for other countries separately, since 2000. As you can see, from 2000 to ...
DOC2 page 1 preview: Supplement A After the Crisis: Outturns, Revisions and Forecast Errors, 2011-2013 Document ID DOC2 Publisher DATA 266 Course Data Desk Edition March 2014 data vintage, published 15...
DOC3 page 1 preview: Briefing B Labour Markets, Prices and Policy Trade-offs after the Crisis Document ID DOC3 Publisher Pacific Policy Institute (policy research group) Date 30 June 2014 Source classi...


## Task 1.1.1 checkpoint

PDF text extraction is complete. The next cells perform only **Task 1.1.2: Chunking**.

## 5. Chunk each page while preserving citation metadata

We use a simple word-based sliding window for this first implementation:

- **300 words per chunk** is a practical starting point that usually leaves room under a 512-token embedding limit; exact token counts will be checked after we select the embedding tokenizer.
- **60 words of overlap** repeats about 20% of the previous chunk so a sentence near a boundary is less likely to lose context.
- **No chunk crosses a page boundary**, so every future answer can retain an exact `DOC#:p#` citation.

We create `all_chunks` for auditability, then create `rag_chunks` by removing every record where `evidence_allowed` is false. Only `rag_chunks` may be sent to the future vector database.

In [7]:
CHUNK_SIZE_WORDS = 300
CHUNK_OVERLAP_WORDS = 60

assert CHUNK_SIZE_WORDS > 0
assert 0 <= CHUNK_OVERLAP_WORDS < CHUNK_SIZE_WORDS


def chunk_page_record(page_record, chunk_size=CHUNK_SIZE_WORDS, overlap=CHUNK_OVERLAP_WORDS):
    words = page_record["text"].split()
    page_chunks = []
    start_word = 0
    chunk_index = 1

    while start_word < len(words):
        end_word = min(start_word + chunk_size, len(words))
        chunk_words = words[start_word:end_word]
        page_chunks.append(
            {
                "chunk_id": (
                    f"{page_record['doc_id']}_p{page_record['page']:03d}_"
                    f"c{chunk_index:03d}"
                ),
                "doc_id": page_record["doc_id"],
                "page": page_record["page"],
                "chunk_index": chunk_index,
                "start_word": start_word,
                "end_word": end_word,
                "word_count": len(chunk_words),
                "trust_level": page_record["trust_level"],
                "evidence_allowed": page_record["evidence_allowed"],
                "text": " ".join(chunk_words),
            }
        )

        if end_word == len(words):
            break

        start_word = end_word - overlap
        chunk_index += 1

    return page_chunks


## 6. Build and validate the chunk collections

The checks below verify unique chunk IDs, complete word coverage for every page, the intended overlap, and the low-trust exclusion rule. They also print a compact summary and one sample chunk.

In [8]:
from collections import Counter

all_chunks = []
for page_record in page_records:
    all_chunks.extend(chunk_page_record(page_record))

chunk_ids = [chunk["chunk_id"] for chunk in all_chunks]
assert len(chunk_ids) == len(set(chunk_ids)), "Chunk IDs must be unique."

for page_record in page_records:
    page_chunks = [
        chunk
        for chunk in all_chunks
        if chunk["doc_id"] == page_record["doc_id"] and chunk["page"] == page_record["page"]
    ]
    assert page_chunks[0]["start_word"] == 0
    assert page_chunks[-1]["end_word"] == len(page_record["text"].split())
    for previous, current in zip(page_chunks, page_chunks[1:]):
        assert current["start_word"] == previous["end_word"] - CHUNK_OVERLAP_WORDS

blocked_chunks = [chunk for chunk in all_chunks if not chunk["evidence_allowed"]]
rag_chunks = [chunk for chunk in all_chunks if chunk["evidence_allowed"]]

assert blocked_chunks, "DOC3 page 7 should produce auditable but blocked chunks."
assert all(
    chunk["doc_id"] == "DOC3"
    and chunk["page"] == 7
    and chunk["trust_level"] == "low"
    for chunk in blocked_chunks
)
assert not any(
    chunk["doc_id"] == "DOC3" and chunk["page"] == 7
    for chunk in rag_chunks
), "DOC3 page 7 must never enter the RAG evidence collection."

all_counts = Counter(chunk["doc_id"] for chunk in all_chunks)
rag_counts = Counter(chunk["doc_id"] for chunk in rag_chunks)

print(f"Chunk size: {CHUNK_SIZE_WORDS} words; overlap: {CHUNK_OVERLAP_WORDS} words")
print(f"All chunks: {len(all_chunks)}")
print(f"RAG evidence chunks: {len(rag_chunks)}")
print(f"Blocked low-trust chunks: {len(blocked_chunks)}")
for doc_id in ("DOC1", "DOC2", "DOC3"):
    print(f"{doc_id}: all={all_counts[doc_id]}, evidence={rag_counts[doc_id]}")

sample_chunk = rag_chunks[0]
sample_preview = sample_chunk["text"][:300]
print("\nSample chunk metadata:")
print({key: value for key, value in sample_chunk.items() if key != "text"})
print(f"Sample text: {sample_preview}...")

Chunk size: 300 words; overlap: 60 words
All chunks: 115
RAG evidence chunks: 114
Blocked low-trust chunks: 1
DOC1: all=101, evidence=101
DOC2: all=7, evidence=7
DOC3: all=7, evidence=6

Sample chunk metadata:
{'chunk_id': 'DOC1_p001_c001', 'doc_id': 'DOC1', 'page': 1, 'chunk_index': 1, 'start_word': 0, 'end_word': 300, 'word_count': 300, 'trust_level': 'high', 'evidence_allowed': True}
Sample text: 1-1 The Crisis Table 1-1 gives you output growth rates for the world economy, for advanced econ- omies and for other countries separately, since 2000. As you can see, from 2000 to 2007 the world economy had a sustained expansion. Annual average world output growth was 3.2%, with advanced economies (...


## Stop here

Task 1.1.2 is complete when the checks above pass. The next step will handle numbered figures and tables; embeddings and vector-database indexing have not started yet.